In [1]:
import os
from openai import OpenAI


BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2:1b"

model = OpenAI(base_url=BASE_URL, api_key="ollama")

def ask_llm(prompt):
    system_prompt = "You are a helpful assistant"
    messages = [{"role":"system", "content":system_prompt}, {"role":"user", "content": prompt}]
    response = model.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content



In [2]:
from ollama import chat
import time

In [3]:
#version 1 of the run_mdoel module
def run_model(model_name, prompt):
    start_time = time.perf_counter()
    response = chat(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    end_time = time.perf_counter()
    latency = end_time - start_time
    generation_time = response.eval_duration/ 1_000_000_000
    tokens_per_second = (response.eval_count/generation_time if generation_time > 0 else 0)

    prompt_eval_time = response.prompt_eval_duration/ 1_000_000_000
    prompt_eval_rate = response.prompt_eval_count/ prompt_eval_time

    return {
    "response": response.message.content,
    "latency": latency,
    "generation_time": generation_time,
    "tokens_per_second": tokens_per_second,
    "output_tokens": response.eval_count,
    "prompt_tokens": response.prompt_eval_count,
    "prompt_tokens_per_second": prompt_eval_rate,
}

In [4]:
#version 2 of the run_model module
import time
import csv
import json
import subprocess
import re
from ollama import chat

def get_model_memory_mb(model_name):
    result = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
    for line in result.stdout.splitlines():
        if model_name in line:
            match = re.search(r'(\d+\.?\d*)\s*(GB|MB)', line)
            if match:
                size, unit = match.groups()
                size = float(size)
                return size * 1024 if unit == "GB" else size
    return None

def run_model(model_name, prompt):
    start_time = time.perf_counter()
    response = chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )
    end_time = time.perf_counter()

    latency = end_time - start_time
    generation_time = response.eval_duration / 1_000_000_000
    tokens_per_second = (response.eval_count / generation_time if generation_time > 0 else 0)

    prompt_eval_time = response.prompt_eval_duration / 1_000_000_000
    prompt_eval_rate = response.prompt_eval_count / prompt_eval_time if prompt_eval_time > 0 else 0

    memory_mb = get_model_memory_mb(model_name)

    return {
        "response": response.message.content,
        "latency": latency,
        "generation_time": generation_time,
        "tokens_per_second": tokens_per_second,
        "output_tokens": response.eval_count,
        "prompt_tokens": response.prompt_eval_count,
        "prompt_tokens_per_second": prompt_eval_rate,
        "memory_mb": memory_mb,
    }

In [5]:
# response = run_model(
#     model_name="llama3.2:3b",
#     prompt="how's weather today?"
# )
# for key, value in response.items():
#     print(f"{key} : {value}")

In [6]:

from IPython.display import Markdown, display
import json

In [7]:
# response = chat(
#         model="llama3.2:3b",
#         messages=[
#             {
#                 "role": "user",
#                 "content": "what is transformer"
#             }
#         ]
#     )
# print(json.dumps(response.model_dump(), indent=4))

In [8]:
# generation_time = response.eval_duration/ 1_000_000_000
# print(generation_time)
# token_per_second = response.eval_count/generation_time
# print(token_per_second)
# prompt_eval_time = response.prompt_eval_duration/ 1_000_000_000
# prompt_eval_rate = response.prompt_eval_duration/ prompt_eval_time


In [9]:

# models = [
#     "llama3.2:3b",
#     "mistral:7b",
#     # "phi4-mini:latest"
# ]

# prompts = [
#     # "Explain recursion.",
#     "Write a bubble sort in Python.",
#     "What is machine learning?"
# ]

# for model in models:
#     display(Markdown(f"\n{'=' * 50}"))
#     display(Markdown(f"Model: {model}"))

#     for i, prompt in enumerate(prompts):
#         display(Markdown(f"Prompt: {prompt}"))
#         try:
#             response = run_model(model, prompt)

#             for key, value in response.items():
#                  display(Markdown(f"{key} : {value}"))

#         except Exception as e:
#             print(f"Error while running {model}:")
#             print(e)
#         print(f"\n{'*' * 50}")

In [10]:
def save_to_csv(result, filepath="results.csv"):
    file_exists = False
    filtered_result = {k : v for k, v in result.items() if k not in ("response", "prompt")}
    try:
        with open(filepath, 'r'):
            file_exists = True
    except FileNotFoundError:
        pass

    with open(filepath, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=filtered_result.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(filtered_result)

        
def save_to_json(results_list, filepath="results.json"):
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(results_list, f, indent=2, ensure_ascii=False)

In [11]:
from pydantic import BaseModel
from typing import Literal

class TicketAnalysis(BaseModel):
    category : Literal["billing", "account", "technical", "other"]
    priority: Literal["low", "medium", "high"]
    summary: str
    requires_escalation: bool

In [12]:
def build_structured_prompt(user_text):
    return f"""Analyze the following text and respond ONLY with valid JSON in this exact format:
{{
  "category": "billing" | "technical" | "account" | "other",
  "priority": "low" | "medium" | "high",
  "summary": "<one sentence>",
  "requires_escalation": true | false
}}

Text: {user_text}

Respond with JSON only. No explanation, no markdown formatting."""

In [ ]:
from pydantic import ValidationError
import json

def get_structured_response(model_name, user_text, max_retries=1):
    prompt = build_structured_prompt(user_text)
    attempt = 0
    

    while attempt <= max_retries:
        result = run_model(model_name, prompt)
        raw_output = result["response"]

        try:
            cleaned = raw_output.strip().strip("```json").strip("```").strip()
            parsed = TicketAnalysis.model_validate_json(cleaned)
            result["valid_json"] = True
            result["parsed"] = parsed.model_dump()
            return result
        except (ValidationError, json.JSONDecodeError) as e:
            attempt += 1
            if attempt > max_retries:
                result["valid_json"] = False
                result["parsed"] = None
                result["error"] = str(e)
                return result
            # reprompt once, mentioning the failure
            prompt = prompt + f"\n\nYour previous response was invalid: {e}. Please fix and respond with valid JSON only."

    return result

In [27]:
def general_prompt(model, prompt):
    result = run_model(model, prompt)
    result["prompt_type"] = "general"
    result["valid_json"] = False
    result["parsed"] = None

    return result


def structured_prompt(model, prompt):
    result = get_structured_response(model, prompt, max_retries=5)

    return result

In [31]:
print(json.dumps(structured_prompt("mistral:7b", "My internet has been down for 3 days and no one from support has responded to my emails." ), indent=4))

{
    "response": " {\n  \"category\": \"technical\",\n  \"priority\": \"high\",\n  \"summary\": \"Internet down for 3 days, no response from support via email.\",\n  \"requires_escalation\": true\n}",
    "latency": 2.513521666987799,
    "generation_time": 2.392649,
    "tokens_per_second": 22.1511805534368,
    "output_tokens": 53,
    "prompt_tokens": 127,
    "prompt_tokens_per_second": 1895.0981123629037,
    "memory_mb": 5017.6,
    "valid_json": true,
    "parsed": {
        "category": "technical",
        "priority": "high",
        "summary": "Internet down for 3 days, no response from support via email.",
        "requires_escalation": true
    }
}


In [35]:
models = [
    "llama3.2:3b",
    "mistral:7b",
    "gemma3:4b "
]

prompts = [
    {"text": "My internet has been down for 3 days...", "type": "structured"},
    {"text": "What is the capital of Australia?", "type": "general"},
    {"text": "Summarize this paragraph: ...", "type": "general"},
]
all_results = []

for model in models:
    for prompt in prompts:
        try:
            if prompt["type"] == "structured":
                response = structured_prompt(model, prompt["text"])
            else:
                response = general_prompt(model, prompt["text"])

            result = {
                "model": model,
                "prompt": prompt["text"],
                **response
            }
            save_to_csv(result)          # writes row-by-row, safe if script crashes mid-run
            all_results.append(result)

        except Exception as e:
            print(f"Error while running {model}:")
            print(e)
            
           
save_to_json(all_results)  
        

In [29]:
import pandas as pd

df = pd.read_csv("/Users/irritatednishantgmail.com/Desktop/SLM App/results.csv")
df.head()

,model,latency,generation_time,tokens_per_second,output_tokens,prompt_tokens,prompt_tokens_per_second,memory_mb,prompt_type,valid_json,parsed
0,llama3.2:3b,12.248306,9.597895,44.280543,425,35,213.066452,2560.0,general,False,NaN
1,llama3.2:3b,0.392840,0.164310,48.688455,8,32,357.565870,2560.0,general,False,NaN
2,llama3.2:3b,0.952092,0.700977,44.223990,31,32,358.615743,2560.0,general,False,NaN


In [ ]:
#print(json.dumps(get_structured_response("llama3.2:3b","explain transformer, 5"), indent=4))

{
    "response": "{\n  \"category\": \"account\",\n  \"priority\": \"low\",\n  \"summary\": \"Explain transformer in simple terms\",\n  \"requires_escalation\": false\n}",
    "latency": 1.074674249990494,
    "generation_time": 0.888118,
    "tokens_per_second": 42.78710711864865,
    "output_tokens": 38,
    "prompt_tokens": 116,
    "prompt_tokens_per_second": 2772.5328043213267,
    "memory_mb": 2560.0,
    "valid_json": true,
    "parsed": {
        "category": "account",
        "priority": "low",
        "summary": "Explain transformer in simple terms",
        "requires_escalation": false
    }
}
